# Speed tests
Here, we test speeds of different libraries for matrix multiplication and kernels:
- numpy
- galois
- bitgauss [documentation](https://bitgauss.readthedocs.io/en/stable/)
- pari
- flint
- torch
- ldpc (sparse GF(2))

Speed results:
- numpy int64 matmul: 2000 x 2000: 22.7s
- galois kernel: 2000 x 2000: 7.33s
- bitgauss kernel:
  - 5000 x 5000: 764ms
  - 10000 x 10000: 7.53s
  - read in from numpy only via lists? -> very slow
- bitgauss Z2 matmul:
  - 5000 x 5000: 500ms
  - 10000 x 10000: 7.3s
- torch int8 matmul:
  - 2000 x 2000: 359ms
  - 5000 x 5000: 11.7s
- flint HNF:
  - 300 x 300: 1.2s
  - 500 x 500: 2.36s
  - 1000 x 1000: 37s
  - timing potentially input dependent as int sizes are dynamic (a random input involves one single column with huge entries)
- Pari int kernel:
  - 100 x 100: 819ms
  - 200 x 200: throws "the PARI stack overflows" error after 38.9s

Bitgauss has can handle ~5 times larger matrices than numpy or galois, corresponding to a speedup factor of ~125

In [1]:
import numpy as np
import z248_linalg as hk
import torch

# galois

In [ ]:
x = np.random.randint(0, 2, size=(2000,2000))
%time y = hk.z2_kernel_galois(x)

# bitgauss

In [17]:
x = np.random.randint(0, 2, size=(250,250))
%time y = hk.z2_kernel_bitgauss(x)

CPU times: user 1.99 ms, sys: 993 μs, total: 2.99 ms
Wall time: 2.54 ms


In [16]:
x = np.random.randint(0, 2, size=(5000,5000))
A = hk.numpy_to_bitgauss(x)
%time B = A @ A

CPU times: user 242 ms, sys: 0 ns, total: 242 ms
Wall time: 242 ms


In [12]:
%time A.gauss(full=True)

CPU times: user 10.1 ms, sys: 1 μs, total: 10.1 ms
Wall time: 10.3 ms


testing reading in and out from numpy or from listlists

In [ ]:
%time bm = bitgauss.BitMatrix.from_int_list(mat.tolist())

CPU times: user 7.53 s, sys: 2.07 s, total: 9.6 s
Wall time: 9.54 s


In [ ]:
%time bm = bitgauss.BitMatrix.from_numpy_bool(mat_bool)
%time mback = bm.to_numpy_bool()b

CPU times: user 528 ms, sys: 20.6 ms, total: 548 ms
Wall time: 544 ms
CPU times: user 303 ms, sys: 109 ms, total: 413 ms
Wall time: 409 ms


# numpy

In [ ]:
x = np.random.randint(0, 2, size=(250,250))
%time y = x @ x

CPU times: user 22.6 s, sys: 10.7 ms, total: 22.6 s
Wall time: 22.7 s


# fmpz

In [ ]:
A = fmpz_mat(np.random.randint(0, 8, size=(1000,1000)).tolist())
%time A.hnf()

# pari

In [ ]:
A = pari.matrix(200, 200, [x for sub in np.random.randint(0, 8, size=(200,200)).tolist() for x in sub])
%time A.matkerint()

# torch

In [15]:
A = torch.randint(-128, 127, (5000, 5000), dtype=torch.int8)
B = torch.randint(-128, 127, (5000, 5000), dtype=torch.int8)
%time C = torch.matmul(A, B)

CPU times: user 9.69 s, sys: 13.4 ms, total: 9.71 s
Wall time: 9.73 s


In [ ]:
%time qC = torch.ops.quantized.matmul(qA, qB)

# ldpc
for sparse matrices

In [2]:
import third_order_gates as tg
import ldpc.mod2 as m2
import scipy.sparse as sp

random sparse matrix

In [53]:
sparsity = 5
nr_rows = 5000
nr_columns = 5000
degrees = np.random.randint(0, sparsity, nr_rows)
nr_entries = np.sum(degrees)
data = np.ones(nr_entries)
indices = np.random.randint(0, nr_columns, nr_entries)
pointers = np.cumsum(degrees)
pointers = np.insert(pointers, 0, 0)
spmat = sp.csr_matrix((data, indices, pointers))

In [54]:
%time kernel = m2.nullspace(spmat)
print(kernel)

CPU times: user 56.1 ms, sys: 1.02 ms, total: 57.1 ms
Wall time: 55.6 ms
<Compressed Sparse Row sparse matrix of dtype 'uint8'
	with 10590 stored elements and shape (1190, 5000)>
  Coords	Values
  (0, 2765)	1
  (1, 1199)	1
  (2, 2308)	1
  (3, 1861)	1
  (4, 720)	1
  (4, 1620)	1
  (4, 1881)	1
  (4, 1910)	1
  (4, 2422)	1
  (4, 2874)	1
  (4, 3721)	1
  (5, 3227)	1
  (6, 1005)	1
  (7, 146)	1
  (8, 2986)	1
  (8, 3761)	1
  (9, 3684)	1
  (9, 3819)	1
  (10, 1406)	1
  (11, 6)	1
  (12, 2249)	1
  (13, 627)	1
  (14, 3824)	1
  (15, 1567)	1
  (16, 2761)	1
  :	:
  (1188, 2287)	1
  (1188, 2523)	1
  (1188, 2582)	1
  (1188, 2684)	1
  (1188, 2693)	1
  (1188, 2698)	1
  (1188, 2985)	1
  (1188, 3114)	1
  (1188, 3166)	1
  (1188, 3241)	1
  (1188, 3311)	1
  (1188, 3369)	1
  (1188, 3442)	1
  (1188, 3856)	1
  (1188, 3939)	1
  (1188, 3978)	1
  (1188, 3996)	1
  (1188, 4004)	1
  (1188, 4184)	1
  (1188, 4393)	1
  (1188, 4470)	1
  (1188, 4687)	1
  (1188, 4847)	1
  (1189, 1203)	1
  (1189, 2759)	1


In [55]:
%time kernel = m2.nullspace(spmat.T)
print(kernel)

CPU times: user 72.6 ms, sys: 995 μs, total: 73.6 ms
Wall time: 71.6 ms
<Compressed Sparse Row sparse matrix of dtype 'uint8'
	with 1773 stored elements and shape (1190, 5000)>
  Coords	Values
  (0, 1385)	1
  (1, 540)	1
  (2, 3042)	1
  (3, 632)	1
  (4, 3814)	1
  (5, 1357)	1
  (5, 1803)	1
  (6, 904)	1
  (6, 1269)	1
  (7, 861)	1
  (8, 3818)	1
  (9, 1401)	1
  (9, 3819)	1
  (10, 1635)	1
  (11, 2038)	1
  (12, 3822)	1
  (13, 3823)	1
  (14, 2706)	1
  (15, 3825)	1
  (16, 3826)	1
  (17, 1707)	1
  (18, 66)	1
  (19, 3829)	1
  (20, 3830)	1
  (21, 3314)	1
  :	:
  (1185, 771)	1
  (1186, 2269)	1
  (1187, 1515)	1
  (1188, 3340)	1
  (1189, 201)	1
  (1189, 606)	1
  (1189, 863)	1
  (1189, 1053)	1
  (1189, 1152)	1
  (1189, 1447)	1
  (1189, 2307)	1
  (1189, 2472)	1
  (1189, 2734)	1
  (1189, 2881)	1
  (1189, 2935)	1
  (1189, 3500)	1
  (1189, 3739)	1
  (1189, 3794)	1
  (1189, 3872)	1
  (1189, 4219)	1
  (1189, 4283)	1
  (1189, 4362)	1
  (1189, 4423)	1
  (1189, 4603)	1
  (1189, 4999)	1


3D toric code check matrix

In [58]:
def spmat_from_listlist(listlist):
    mat_indices = np.hstack([np.array(x, dtype=int) for x in listlist])
    mat_pointers = np.cumsum([len(x) for x in listlist])
    mat_pointers = np.insert(mat_pointers, 0, 0)
    mat_data = np.ones((mat_pointers[-1]))
    return sp.csr_matrix((mat_data, mat_indices, mat_pointers))

In [59]:
# calculating Z stabilizers and logicals of 3D toric code (commutant of X stabilizers)
tc_3d = tg.ti_transversal_gate_finder(3, 3)
tc_3d.add_checks([[([0,0,0], 0), ([0,0,0], 1), ([0,0,0], 2), ([-1,0,0], 0), ([0,-1,0], 1), ([0,0,-1], 2)]])
tc_3d.add_other_checks([[([0,0,0], 0), ([0,0,0], 1), ([0,1,0], 0), ([1,0,0], 1)], [([0,0,0], 0), ([0,0,0], 2), ([0,0,1], 0), ([1,0,0], 2)], [([0,0,0], 1), ([0,0,0], 2), ([0,0,1], 1), ([0,1,0], 2)]])
tc_3d_finite = tc_3d.as_finite_code(np.array([[30,0,0],[0,30,0],[0,0,30]]))
spmat = spmat_from_listlist(tc_3d_finite.checks)

In [ ]:
spmat = spmat_from_listlist(tc_3d_finite.checks)
print(spmat.shape)
%time kernel = m2.nullspace(spmat)
print(kernel.shape)

(27000, 81000)
CPU times: user 4.08 s, sys: 37.1 ms, total: 4.12 s
Wall time: 4.1 s
(54001, 81000)


In [61]:
spmat = spmat_from_listlist(tc_3d_finite.checks).T
print(spmat.shape)
%time kernel = m2.nullspace(spmat)
print(kernel.shape)

(81000, 27000)
CPU times: user 3.44 s, sys: 16 ms, total: 3.45 s
Wall time: 3.43 s
(1, 27000)


In [62]:
spmat = spmat_from_listlist(tc_3d_finite.other_checks)
print(spmat.shape)
%time kernel = m2.nullspace(spmat)
print(kernel.shape)

(81000, 81000)
CPU times: user 10.1 s, sys: 45.2 ms, total: 10.1 s
Wall time: 10.1 s
(27002, 81000)


In [63]:
spmat = spmat_from_listlist(tc_3d_finite.other_checks).T
print(spmat.shape)
%time kernel = m2.nullspace(spmat)
print(kernel.shape)

(81000, 81000)
CPU times: user 11.5 s, sys: 49.1 ms, total: 11.5 s
Wall time: 11.5 s
(27002, 81000)


In [ ]:
#bitgauss for comparison
# 5-10x slower on 20 x 20 x 20 3D TC, dies on 30 x 30 x 30 because of memory overflow
mat = tg.transversal_gate_finder.matrix_from_list(tc_3d_finite.nr_qubits, tc_3d_finite.checks).T
%time kernel = hk.z2_kernel_bitgauss(mat)
print(kernel.shape)